# 02 — Full Dataset Preparation

**Goal:** Merge all data sources into a single contract-level dataset.  
Each row = one option contract × one trading day × one stock, enriched with price features, fundamentals, and greeks.

## Merge Strategy
```
OPTIONS (1,854,022 rows)          <- base
  ↓ filter call options only
  ↓ join daily prices             on (symbol, trade_date == date)
  ↓ engineer price features       returns, momentum, volatility, MA, drawdown, volume
  ↓ null audit + clean fundamentals
  ↓ engineer fundamental features margins, growth, ratios
  ↓ join fundamentals             on (symbol, nearest quarter <= trade_date) via merge_asof
  ↓ join overview                 on (symbol) -- static broadcast
  ↓ engineer options features     DTE, moneyness, mid, spread
  ↓ bucket classification         9 buckets: ATM/OTM5/OTM10 x 30/60/90
  ↓ target label                  optimal_bucket via Sharpe across buckets
  ↓ save -> data/final/modeling_dataset.parquet
```

## Cell 0 — Imports & Configuration

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.4f}'.format)

TICKERS    = ['ADMA','NTRA','AXON','SHAK','AAPL','MSFT','NVDA','AMZN','GOOG','META']
DATE_START = '2015-01-01'
DATE_END   = '2025-12-31'
RISK_FREE  = 0.05

PROC  = Path('../data')
FINAL = Path('../data/final')
FINAL.mkdir(parents=True, exist_ok=True)

VALID_BUCKETS = [f'{m}_{d}' for m in ['ATM','OTM5','OTM10'] for d in [30,60,90]]
BUCKET_TO_INT = {b: i for i, b in enumerate(VALID_BUCKETS)}
print(f'Buckets ({len(VALID_BUCKETS)}): {VALID_BUCKETS}')
print('Imports OK')

Buckets (9): ['ATM_30', 'ATM_60', 'ATM_90', 'OTM5_30', 'OTM5_60', 'OTM5_90', 'OTM10_30', 'OTM10_60', 'OTM10_90']
Imports OK


## Cell 1 — Load All Raw Data

In [2]:
def load_parquet(path, label):
    df = pd.read_parquet(path)
    print(f'  {label}: {df.shape}')
    return df

print('Loading raw data...')
daily_raw    = load_parquet(PROC / 'daily_adjusted/ALL_daily_adjusted.parquet',  'Daily prices')
income_raw   = load_parquet(PROC / 'fundamentals/ALL_income_statement.parquet',  'Income statement')
balance_raw  = load_parquet(PROC / 'fundamentals/ALL_balance_sheet.parquet',     'Balance sheet')
cashflow_raw = load_parquet(PROC / 'fundamentals/ALL_cash_flow.parquet',         'Cash flow')
options_raw  = load_parquet(PROC / 'options/ALL_options.parquet',                'Options')
overview_raw = pd.read_csv(PROC / 'fundamentals/ALL_overview.csv')
print(f'  Overview: {overview_raw.shape}')

# Parse dates
daily_raw['date']          = pd.to_datetime(daily_raw['date'])
options_raw['trade_date']  = pd.to_datetime(options_raw['trade_date'])
options_raw['expiration']  = pd.to_datetime(options_raw['expiration'])
for df in [income_raw, balance_raw, cashflow_raw]:
    df['fiscalDateEnding'] = pd.to_datetime(df['fiscalDateEnding'])

print('\nAll data loaded and dates parsed.')

Loading raw data...
  Daily prices: (27516, 10)
  Income statement: (736, 27)
  Balance sheet: (729, 39)
  Cash flow: (731, 30)
  Options: (1854022, 20)
  Overview: (10, 55)

All data loaded and dates parsed.


### Data Cleaning

In [3]:
# Null value audit across all raw files 
def null_audit(df, label):
    total     = len(df)
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0].sort_values(ascending=False)
    print(f"\n{'='*55}")
    print(f"{label}  —  {df.shape[0]:,} rows * {df.shape[1]} cols")
    print(f"{'='*55}")
    if null_cols.empty:
        print("  ✓ No null values found")
    else:
        print(f"  {len(null_cols)} columns have nulls:\n")
        pct = (null_cols / total * 100).round(1)
        summary = pd.DataFrame({'null_count': null_cols, 'null_pct': pct})
        print(summary.to_string())

null_audit(daily_raw,    "Daily Adjusted")
null_audit(options_raw,  "Options")
null_audit(income_raw,   "Income Statement")
null_audit(balance_raw,  "Balance Sheet")
null_audit(cashflow_raw, "Cash Flow")
null_audit(overview_raw, "Overview")


Daily Adjusted  —  27,516 rows * 10 cols
  ✓ No null values found

Options  —  1,854,022 rows * 20 cols
  ✓ No null values found

Income Statement  —  736 rows * 27 cols
  20 columns have nulls:

                                   null_count  null_pct
nonInterestIncome                         736  100.0000
comprehensiveIncomeNetOfTax               736  100.0000
interestAndDebtExpense                    736  100.0000
investmentIncomeNet                       736  100.0000
depreciation                              736  100.0000
interestIncome                            508   69.0000
netInterestIncome                         468   63.6000
otherNonOperatingIncome                   380   51.6000
netIncomeFromContinuingOperations          63    8.6000
researchAndDevelopment                     48    6.5000
interestExpense                            29    3.9000
depreciationAndAmortization                22    3.0000
totalRevenue                               20    2.7000
grossProfit        

In [4]:
# Drop 100% null columns from each source
income_drop = ['nonInterestIncome','comprehensiveIncomeNetOfTax',
               'interestAndDebtExpense','investmentIncomeNet','depreciation']

balance_drop = ['longTermDebtNoncurrent','otherNonCurrentAssets','deferredRevenue',
                'currentDebt','investments','accumulatedDepreciationAmortizationPPE']

cashflow_drop = ['paymentsForRepurchaseOfCommonStock','paymentsForRepurchaseOfPreferredStock',
                 'proceedsFromOperatingActivities','changeInOperatingLiabilities',
                 'changeInOperatingAssets','proceedsFromSaleOfTreasuryStock',
                 'proceedsFromIssuanceOfPreferredStock',
                 'proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet',
                 'profitLoss','proceedsFromIssuanceOfCommonStock',
                 'dividendPayoutPreferredStock','proceedsFromRepaymentsOfShortTermDebt',
                 'paymentsForOperatingActivities','paymentsForRepurchaseOfEquity']

# Safe drop — only drop if column exists
income_raw   = income_raw.drop(columns=[c for c in income_drop   if c in income_raw.columns])
balance_raw  = balance_raw.drop(columns=[c for c in balance_drop  if c in balance_raw.columns])
cashflow_raw = cashflow_raw.drop(columns=[c for c in cashflow_drop if c in cashflow_raw.columns])
print('100% null columns dropped.')

100% null columns dropped.


In [5]:
def clean_fundamentals(df, label):
    key_cols = ['symbol', 'fiscalDateEnding', 'reportedCurrency']
    fill_cols = df.columns.difference(key_cols)
    
    df = df.sort_values(['symbol', 'fiscalDateEnding'])
    
    # Forward-fill then backward-fill within each symbol
    df[fill_cols] = (df.groupby('symbol')[fill_cols]
                       .transform(lambda x: x.ffill().bfill()))
    
    # For any still-remaining nulls, fill with column median across all symbols
    for col in fill_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    remaining = df.isnull().sum().sum()
    print(f"{label}: {remaining} nulls remaining after clean")
    return df

income_raw   = clean_fundamentals(income_raw,   "Income Statement")
balance_raw  = clean_fundamentals(balance_raw,  "Balance Sheet")
cashflow_raw = clean_fundamentals(cashflow_raw, "Cash Flow")

Income Statement: 0 nulls remaining after clean
Balance Sheet: 0 nulls remaining after clean
Cash Flow: 0 nulls remaining after clean


In [6]:
overview_raw['DividendPerShare'] = overview_raw['DividendPerShare'].fillna(0)
overview_raw['DividendYield']    = overview_raw['DividendYield'].fillna(0)
overview_raw['DividendDate']     = overview_raw['DividendDate'].fillna('N/A')
overview_raw['ExDividendDate']   = overview_raw['ExDividendDate'].fillna('N/A')
overview_raw['PERatio']          = overview_raw['PERatio'].fillna(overview_raw['PERatio'].median())
print("Overview nulls filled.")

Overview nulls filled.


In [7]:
for df, label in [(income_raw,'Income'), (balance_raw,'Balance'),
                  (cashflow_raw,'Cashflow'), (overview_raw,'Overview')]:
    remaining = df.isnull().sum().sum()
    print(f"{label}: {remaining} nulls remaining")

Income: 0 nulls remaining
Balance: 0 nulls remaining
Cashflow: 0 nulls remaining
Overview: 0 nulls remaining


## Cell 2 — Price Feature Engineering
Engineer all features from daily_adjusted. Rename `volume` → `stock_volume` to avoid collision with options volume.

In [8]:
def engineer_price_features(df):
    df = df.sort_values(['symbol', 'date']).copy()
    
    # Rename volume upfront to avoid collision with options volume
    df = df.rename(columns={'volume': 'stock_volume'})
    
    g  = df.groupby('symbol', group_keys=False)

    # Returns
    df['return_1d']   = g['adj_close'].transform(lambda x: x.pct_change())
    df['log_return']  = g['adj_close'].transform(lambda x: np.log(x / x.shift(1)))

    # Momentum
    df['momentum_1m']  = g['adj_close'].transform(lambda x: x / x.shift(21)  - 1)
    df['momentum_3m']  = g['adj_close'].transform(lambda x: x / x.shift(63)  - 1)
    df['momentum_6m']  = g['adj_close'].transform(lambda x: x / x.shift(126) - 1)
    df['momentum_12m'] = g['adj_close'].transform(lambda x: x / x.shift(252) - 1)

    # Moving averages
    df['ma_20']          = g['adj_close'].transform(lambda x: x.rolling(20).mean())
    df['ma_50']          = g['adj_close'].transform(lambda x: x.rolling(50).mean())
    df['ma_200']         = g['adj_close'].transform(lambda x: x.rolling(200).mean())
    df['price_vs_ma20']  = df['adj_close'] / df['ma_20']  - 1
    df['price_vs_ma50']  = df['adj_close'] / df['ma_50']  - 1
    df['price_vs_ma200'] = df['adj_close'] / df['ma_200'] - 1
    df['ma50_vs_ma200']  = df['ma_50']     / df['ma_200'] - 1

    # Volatility
    df['vol_20d']  = g['log_return'].transform(lambda x: x.rolling(20).std()  * np.sqrt(252))
    df['vol_60d']  = g['log_return'].transform(lambda x: x.rolling(60).std()  * np.sqrt(252))
    df['vol_120d'] = g['log_return'].transform(lambda x: x.rolling(120).std() * np.sqrt(252))
    df['intraday_range'] = (df['high'] - df['low']) / df['adj_close']
    df['_hl_sq']   = (np.log(df['high'] / df['low'])) ** 2
    df['parkinson_vol_20d'] = g['_hl_sq'].transform(
        lambda x: np.sqrt(x.rolling(20).mean() / (4 * np.log(2))) * np.sqrt(252))
    df.drop(columns=['_hl_sq'], inplace=True)

    # Drawdown
    df['rolling_max']     = g['adj_close'].transform(lambda x: x.rolling(252, min_periods=1).max())
    df['drawdown']        = df['adj_close'] / df['rolling_max'] - 1
    df['drawdown_regime'] = (df['drawdown'] < -0.20).astype(int)
    df['near_52w_high']   = df['adj_close'] / df['rolling_max']

    # Volume — all refs now use stock_volume
    df['volume_ma20']   = g['stock_volume'].transform(lambda x: x.rolling(20).mean())
    df['volume_spike']  = df['stock_volume'] / df['volume_ma20']
    df['volume_change'] = g['stock_volume'].transform(lambda x: x.pct_change())

    return df

In [9]:
print('Engineering price features...')
daily_feat = engineer_price_features(daily_raw)
print(f'Daily features shape: {daily_feat.shape}')
print(f'New columns added: {daily_feat.shape[1] - daily_raw.shape[1]}')
daily_feat.head(2)

Engineering price features...
Daily features shape: (27516, 35)
New columns added: 25


,date,symbol,open,high,low,close,adj_close,stock_volume,dividend,split_coeff,return_1d,log_return,momentum_1m,momentum_3m,momentum_6m,momentum_12m,ma_20,ma_50,ma_200,price_vs_ma20,price_vs_ma50,price_vs_ma200,ma50_vs_ma200,vol_20d,vol_60d,vol_120d,intraday_range,parkinson_vol_20d,rolling_max,drawdown,drawdown_regime,near_52w_high,volume_ma20,volume_spike,volume_change
10920,2015-01-02,AAPL,111.3900,111.4400,107.3500,109.3300,24.2152,53204626,0.0000,1.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1689,NaN,24.2152,0.0000,0,1.0000,NaN,NaN,NaN
10921,2015-01-05,AAPL,108.2900,108.6500,105.4100,106.2500,23.5330,64285491,0.0000,1.0000,-0.0282,-0.0286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1377,NaN,24.2152,-0.0282,0,0.9718,NaN,NaN,0.2083


## Cell 3 — Fundamental Feature Engineering
Compute ratios from income statement, balance sheet, and cash flow.

In [10]:
def to_numeric_cols(df, exclude=['symbol','fiscalDateEnding','reportedCurrency']):
    cols = [c for c in df.columns if c not in exclude]
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
    return df

income_raw   = to_numeric_cols(income_raw)
balance_raw  = to_numeric_cols(balance_raw)
cashflow_raw = to_numeric_cols(cashflow_raw)

# Income statement features 
inc = income_raw[['symbol','fiscalDateEnding','grossProfit','totalRevenue',
                   'operatingIncome','netIncome','ebit','ebitda','interestExpense']].copy()
inc = inc.sort_values(['symbol','fiscalDateEnding'])
g   = inc.groupby('symbol')

inc['gross_margin']        = inc['grossProfit']     / inc['totalRevenue']
inc['operating_margin']    = inc['operatingIncome'] / inc['totalRevenue']
inc['net_margin']          = inc['netIncome']       / inc['totalRevenue']
inc['ebitda_margin']       = inc['ebitda']          / inc['totalRevenue']
inc['revenue_growth_qoq']  = g['totalRevenue'].transform(lambda x: x.pct_change(1))
inc['revenue_growth_yoy']  = g['totalRevenue'].transform(lambda x: x.pct_change(4))
inc['earnings_growth_yoy'] = g['netIncome'].transform(lambda x: x.pct_change(4))

inc_feat = inc[['symbol','fiscalDateEnding','gross_margin','operating_margin',
                'net_margin','ebitda_margin','revenue_growth_qoq',
                'revenue_growth_yoy','earnings_growth_yoy','totalRevenue','netIncome']]

# Balance sheet features
bal = balance_raw[['symbol','fiscalDateEnding','totalAssets','totalCurrentAssets',
                    'totalCurrentLiabilities','totalLiabilities','totalShareholderEquity',
                    'longTermDebt','commonStockSharesOutstanding']].copy()
bal = bal.sort_values(['symbol','fiscalDateEnding'])

ni_lookup = inc[['symbol','fiscalDateEnding','netIncome','totalRevenue']]
bal2 = bal.merge(ni_lookup, on=['symbol','fiscalDateEnding'], how='left')

bal['current_ratio']  = bal['totalCurrentAssets']  / bal['totalCurrentLiabilities']
bal['debt_to_equity'] = bal['totalLiabilities']     / bal['totalShareholderEquity']
bal['roe']            = bal2['netIncome']            / bal['totalShareholderEquity']
bal['roa']            = bal2['netIncome']            / bal['totalAssets']
bal['asset_turnover'] = bal2['totalRevenue']         / bal['totalAssets']

bal_feat = bal[['symbol','fiscalDateEnding','current_ratio','debt_to_equity',
                'roe','roa','asset_turnover','totalAssets',
                'totalShareholderEquity','longTermDebt','commonStockSharesOutstanding']]

# Cash flow features 
cf = cashflow_raw[['symbol','fiscalDateEnding','operatingCashflow','capitalExpenditures']].copy()
cf = cf.sort_values(['symbol','fiscalDateEnding'])
cf['operatingCashflow']   = pd.to_numeric(cf['operatingCashflow'],   errors='coerce')
cf['capitalExpenditures'] = pd.to_numeric(cf['capitalExpenditures'],  errors='coerce')
cf['fcf']         = cf['operatingCashflow'] - cf['capitalExpenditures'].abs()
cf['capex_ratio'] = cf['capitalExpenditures'].abs() / cf['operatingCashflow'].abs()
cf2 = cf.merge(ni_lookup[['symbol','fiscalDateEnding','totalRevenue']],
               on=['symbol','fiscalDateEnding'], how='left')
cf['ocf_margin']  = cf['operatingCashflow'] / cf2['totalRevenue']

cf_feat = cf[['symbol','fiscalDateEnding','fcf','capex_ratio','ocf_margin','operatingCashflow']]

print(f'Income features:   {inc_feat.shape}')
print(f'Balance features:  {bal_feat.shape}')
print(f'Cashflow features: {cf_feat.shape}')

Income features:   (736, 11)
Balance features:  (729, 11)
Cashflow features: (731, 6)


## Cell 4 — Merge Fundamentals Into One Quarterly Table

In [11]:
fund_feat = inc_feat.merge(bal_feat, on=['symbol','fiscalDateEnding'], how='outer')
fund_feat = fund_feat.merge(cf_feat, on=['symbol','fiscalDateEnding'], how='outer')
fund_feat = fund_feat.sort_values(['symbol','fiscalDateEnding']).reset_index(drop=True)

print(f'Combined fundamentals: {fund_feat.shape}')
fund_feat.head(3)

Combined fundamentals: (736, 24)


,symbol,fiscalDateEnding,gross_margin,operating_margin,net_margin,ebitda_margin,revenue_growth_qoq,revenue_growth_yoy,earnings_growth_yoy,totalRevenue,netIncome,current_ratio,debt_to_equity,roe,roa,asset_turnover,totalAssets,totalShareholderEquity,longTermDebt,commonStockSharesOutstanding,fcf,capex_ratio,ocf_margin,operatingCashflow
0,AAPL,2005-12-31,0.2720,0.1305,0.0983,0.1395,NaN,NaN,NaN,5749000000.0000,565000000,2.3105,0.6922,0.0001,0.0001,0.0017,14181000000.0000,8380000000.0000,0.0000,24477796000.0000,201000000.0000,0.2898,11.6064,283000000.0000
1,AAPL,2006-03-31,0.2975,0.1214,0.0941,0.1328,-0.2418,NaN,NaN,4359000000.0000,410000000,2.4080,0.6023,-0.0007,-0.0004,0.0015,13911000000.0000,8682000000.0000,0.0000,24599036000.0000,-318000000.0000,1.5440,-5.8603,-125000000.0000
2,AAPL,2006-06-30,0.3032,0.1295,0.1080,0.1426,0.0025,NaN,NaN,4370000000.0000,472000000,2.2867,0.6199,0.0004,0.0003,0.0017,15114000000.0000,9330000000.0000,0.0000,24538304000.0000,770000000.0000,0.2354,39.2724,1007000000.0000


## Cell 5 — Build Contract-Level Dataset
**Base = OPTIONS**. Filter to call options only, then join daily price features.

In [12]:
# Filter to call options only (covered calls sell calls)
print(f'Total options rows: {len(options_raw):,}')
calls = options_raw[options_raw['call_put'] == 'call'].copy()
print(f'Call options only:  {len(calls):,}')

# Numeric cast
num_cols = ['strike','last','mark','bid','bid_size','ask','ask_size',
            'volume','open_interest','implied_vol','delta','gamma','theta','vega','rho']
for col in num_cols:
    calls[col] = pd.to_numeric(calls[col], errors='coerce')

# Rename date in daily_feat for join
daily_for_join = daily_feat.rename(columns={'date': 'trade_date'})

# Join price features onto options
dataset = calls.merge(
    daily_for_join,
    on=['symbol', 'trade_date'],
    how='inner'
)

print(f'\nAfter joining daily prices: {dataset.shape}')
print(f'Rows dropped (no price match): {len(calls) - len(dataset):,}')

Total options rows: 1,854,022
Call options only:  927,008

After joining daily prices: (927008, 53)
Rows dropped (no price match): 0


## Cell 6 — Point-in-Time Fundamental Join
Use `merge_asof` — joins most recent quarter available on each trade_date. No lookahead bias.

In [13]:
# merge_asof requires GLOBAL sort by the join key (not by symbol+date)
dataset   = dataset.sort_values('trade_date').reset_index(drop=True)
fund_feat = fund_feat.sort_values('fiscalDateEnding').reset_index(drop=True)

# Ensure datetime dtype and drop any null join keys
dataset['trade_date']         = pd.to_datetime(dataset['trade_date'])
fund_feat['fiscalDateEnding'] = pd.to_datetime(fund_feat['fiscalDateEnding'])
dataset   = dataset.dropna(subset=['trade_date'])
fund_feat = fund_feat.dropna(subset=['fiscalDateEnding'])

dataset = pd.merge_asof(
    dataset,
    fund_feat,
    left_on   = 'trade_date',
    right_on  = 'fiscalDateEnding',
    by        = 'symbol',
    direction = 'backward'
)

print(f'After fundamental join: {dataset.shape}')

After fundamental join: (927008, 76)


## Cell 7 — Overview Static Join

In [14]:
# Rename capital-S Symbol to lowercase to match dataset
overview_raw = overview_raw.rename(columns={'Symbol': 'symbol'})

overview_cols = ['symbol','Sector','Industry','MarketCapitalization',
                 'Beta','PERatio','PBRatio','DividendYield',
                 '52WeekHigh','52WeekLow','AnalystTargetPrice','EPS',
                 'ProfitMargin','ReturnOnAssetsTTM','ReturnOnEquityTTM',
                 'QuarterlyEarningsGrowthYOY','QuarterlyRevenueGrowthYOY',
                 'EVToEBITDA','PriceToBookRatio']
overview_cols = [c for c in overview_cols if c in overview_raw.columns]
ov = overview_raw[overview_cols].copy()

dataset = dataset.merge(ov, on='symbol', how='left')
print(f'After overview join: {dataset.shape}')
print(f'Overview cols added: {len(overview_cols) - 1}')

After overview join: (927008, 93)
Overview cols added: 17


## Cell 8 — Options Feature Engineering
Compute DTE, moneyness, mid price, spread, and premium yield.

In [15]:
dataset['dte']           = (dataset['expiration'] - dataset['trade_date']).dt.days
dataset['moneyness']     = dataset['strike'] / dataset['adj_close']
dataset['log_moneyness'] = np.log(dataset['moneyness'])
dataset['otm_pct']       = (dataset['strike'] - dataset['adj_close']) / dataset['adj_close'] * 100
dataset['mid_price']     = (dataset['bid'] + dataset['ask']) / 2
dataset['spread']        = dataset['ask'] - dataset['bid']
dataset['spread_pct']    = dataset['spread'] / dataset['mid_price'].replace(0, np.nan)
dataset['premium_yield'] = dataset['mid_price'] / dataset['adj_close']

print(f'Options features added.')
print(f"DTE range:       {dataset['dte'].min():.0f} — {dataset['dte'].max():.0f} days")
print(f"Moneyness range: {dataset['moneyness'].min():.3f} — {dataset['moneyness'].max():.3f}")

Options features added.
DTE range:       0 — 1052 days
Moneyness range: 0.002 — 303.762


## Cell 9 — Bucket Classification
Filter to covered call universe (-10% to +30% OTM), then assign each contract to one of 9 buckets.

| Bucket  | otm_pct range  | DTE range  |
|---------|----------------|------------|
| ATM     | -10% to +5%    | —          |
| OTM5    | +5% to +15%    | —          |
| OTM10   | +15% to +30%   | —          |
| _30     | —              | 1–60d      |
| _60     | —              | 60–100d    |
| _90     | —              | 100–500d   |

In [16]:
dataset.head(20)

,contractID,symbol,expiration,strike,call_put,last,mark,bid,bid_size,ask,ask_size,volume,open_interest,trade_date,implied_vol,delta,gamma,theta,vega,rho,open,high,low,close,adj_close,stock_volume,dividend,split_coeff,return_1d,log_return,...,commonStockSharesOutstanding,fcf,capex_ratio,ocf_margin,operatingCashflow,Sector,Industry,MarketCapitalization,Beta,PERatio,DividendYield,52WeekHigh,52WeekLow,AnalystTargetPrice,EPS,ProfitMargin,ReturnOnAssetsTTM,ReturnOnEquityTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,EVToEBITDA,PriceToBookRatio,dte,moneyness,log_moneyness,otm_pct,mid_price,spread,spread_pct,premium_yield
0,AAPL150206C00126000,AAPL,2015-02-06,126.0000,call,0.0200,0.0100,0.0100,387,0.0200,100,1381,2471,2015-02-02,0.2490,0.0108,0.0092,-0.0110,0.0035,0.0001,118.0500,119.1700,116.0800,118.6300,26.2750,62739100,0.0000,1.0000,0.0125,0.0125,...,23527212000.0000,30457000000.0000,0.0968,202.6051,33722000000.0000,TECHNOLOGY,CONSUMER ELECTRONICS,3825723245000,1.1160,33.2400,0.0039,288.3500,168.4800,292.1500,7.8300,0.2700,0.2440,1.5200,0.1830,0.1570,25.1500,43.3300,4,4.7954,1.5677,379.5434,0.0150,0.0100,0.6667,0.0006
1,NVDA150918C00021000,NVDA,2015-09-18,21.0000,call,1.0500,1.1900,1.1600,27,1.2100,215,100,57,2015-02-02,0.2783,0.4224,0.0907,-0.0037,0.0607,0.0444,19.3200,19.7000,18.9400,19.6200,0.4709,6490149,0.0000,1.0000,0.0216,0.0214,...,22262920000.0000,411684000.0000,0.0701,0.0092,442729000.0000,TECHNOLOGY,SEMICONDUCTORS,4456078901000,2.3750,37.4200,0.0002,212.1800,86.6000,265.1800,4.9000,0.5560,0.5120,1.0150,0.9560,0.7320,30.4600,28.3200,228,44.5977,3.7977,4359.7673,1.1850,0.0500,0.0422,2.5166
2,NVDA150918C00020000,NVDA,2015-09-18,20.0000,call,0.8300,1.5800,1.5600,27,1.6100,206,0,58,2015-02-02,0.2783,0.5104,0.0924,-0.0038,0.0618,0.0528,19.3200,19.7000,18.9400,19.6200,0.4709,6490149,0.0000,1.0000,0.0216,0.0214,...,22262920000.0000,411684000.0000,0.0701,0.0092,442729000.0000,TECHNOLOGY,SEMICONDUCTORS,4456078901000,2.3750,37.4200,0.0002,212.1800,86.6000,265.1800,4.9000,0.5560,0.5120,1.0150,0.9560,0.7320,30.4600,28.3200,228,42.4740,3.7489,4147.3974,1.5850,0.0500,0.0315,3.3661
3,NVDA150918C00019000,NVDA,2015-09-18,19.0000,call,2.0000,2.0800,2.0500,21,2.1000,175,2,20,2015-02-02,0.2880,0.6019,0.0864,-0.0038,0.0598,0.0607,19.3200,19.7000,18.9400,19.6200,0.4709,6490149,0.0000,1.0000,0.0216,0.0214,...,22262920000.0000,411684000.0000,0.0701,0.0092,442729000.0000,TECHNOLOGY,SEMICONDUCTORS,4456078901000,2.3750,37.4200,0.0002,212.1800,86.6000,265.1800,4.9000,0.5560,0.5120,1.0150,0.9560,0.7320,30.4600,28.3200,228,40.3503,3.6976,3935.0275,2.0750,0.0500,0.0241,4.4067
4,NVDA150918C00018000,NVDA,2015-09-18,18.0000,call,1.2700,2.6700,2.6400,13,2.7000,186,0,2,2015-02-02,0.2978,0.6869,0.0767,-0.0036,0.0549,0.0673,19.3200,19.7000,18.9400,19.6200,0.4709,6490149,0.0000,1.0000,0.0216,0.0214,...,22262920000.0000,411684000.0000,0.0701,0.0092,442729000.0000,TECHNOLOGY,SEMICONDUCTORS,4456078901000,2.3750,37.4200,0.0002,212.1800,86.6000,265.1800,4.9000,0.5560,0.5120,1.0150,0.9560,0.7320,30.4600,28.3200,228,38.2266,3.6435,3722.6577,2.6700,0.0600,0.0225,5.6703
5,NVDA150918C00017000,NVDA,2015-09-18,17.0000,call,1.6000,3.3500,3.3000,105,3.4000,164,0,6,2015-02-02,0.2978,0.7673,0.0662,-0.0031,0.0474,0.0732,19.3200,19.7000,18.9400,19.6200,0.4709,6490149,0.0000,1.0000,0.0216,0.0214,...,22262920000.0000,411684000.0000,0.0701,0.0092,442729000.0000,TECHNOLOGY,SEMICONDUCTORS,4456078901000,2.3750,37.4200,0.0002,212.1800,86.6000,265.1800,4.9000,0.5560,0.5120,1.0150,0.9560,0.7320,30.4600,28.3200,228,36.1029,3.5864,3510.2878,3.3500,0.1000,0.0299,7.1144
6,NVDA150918C00016000,NVDA,2015-09-18,16.0000,call,0.0000,4.1000,4.0000,365,4.2000,308,0,0,2015-02-02,0.3075,0.8324,0.0526,-0.0027,0.0389,0.0763,19.3200,19.7000,18.9400,19.6200,0.4709,6490149,0.0000,1.0000,0.0216,0.0214,...,22262920000.0000,411684000.0000,0.0701,0.0092,442729000.0000,TECHNOLOGY,SEMICONDUCTORS,4456078901000,2.3750,37.4200,0.0002,212.1800,86.6000,265.1800,4.9000,0.5560,0.5120,1.0150,0.9560,0.7320,30

In [17]:
dataset.describe()

,expiration,strike,last,mark,bid,bid_size,ask,ask_size,volume,open_interest,trade_date,implied_vol,delta,gamma,theta,vega,rho,open,high,low,close,adj_close,stock_volume,dividend,split_coeff,return_1d,log_return,momentum_1m,momentum_3m,momentum_6m,...,totalShareholderEquity,longTermDebt,commonStockSharesOutstanding,fcf,capex_ratio,ocf_margin,operatingCashflow,MarketCapitalization,Beta,PERatio,DividendYield,52WeekHigh,52WeekLow,AnalystTargetPrice,EPS,ProfitMargin,ReturnOnAssetsTTM,ReturnOnEquityTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,EVToEBITDA,PriceToBookRatio,dte,moneyness,log_moneyness,otm_pct,mid_price,spread,spread_pct,premium_yield
count,927008,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,924027.0000,918175.0000,908851.0000,...,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,927008.0000,925688.0000,927008.0000
mean,2021-12-15 11:11:52.009389056,783.9387,98.5669,148.2327,146.6285,40.9336,149.8744,49.0112,195.1004,1121.3616,2021-06-24 16:57:21.399426560,0.4364,0.5864,0.0059,-0.2026,0.9327,1.3599,828.7684,838.8387,818.0781,829.1625,152.2784,32529985.7384,0.0003,1.0000,0.0031,0.0028,0.0338,0.1003,0.2217,...,112502752055.4999,28688117411.7325,12408720638.0184,8557456929.9445,0.5677,NaN,14364314886.7820,2945310975632.3374,1.4639,40.8161,0.0022,356.9213,193.4231,379.8403,9.4192,0.2865,0.1934,0.5399,0.3236,0.2731,24.3567,15.3072,173.7601,10.1420,1.3533,914.2014,148.2514,3.2459,0.2691,1.8188
min,2015-02-06 00:00:00,0.5000,0.0000,0.0100,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2015-02-02 00:00:00,0.0149,0.0000,0.0000,-21.5896,0.0000,0.0000,1.1400,1.2100,1.1200,1.1900,0.4709,71189.0000,0.0000,1.0000,-0.1510,-0.1637,-0.4168,-0.5444,-0.6722,...,-197774000.0000,0.0000,13339000.0000,-17741000000.0000,0.0073,-inf,-2790000000.0000,3746502000.0000,0.5930,18.2600,0.0000,25.6700,13.7600,25.6700,-1.5200,-0.0903,-0.0954,-0.1430,-0.9800,0.1360,-26.8000,5.7200,0.0000,0.0024,-6.0252,-99.7583,0.0000,0.0000,0.0000,0.0000
25%,2019-08-16 00:00:00,138.0000,0.1900,3.9000,3.5000,1.0000,4.4000,3.0000,0.0000,2.0000,2019-05-01 00:00:00,0.2685,0.1832,0.0002,-0.1814,0.0204,0.0348,156.2900,158.4200,154.6700,157.9600,49.9458,2841976.0000,0.0000,1.0000,-0.0081,-0.0081,-0.0345,-0.0328,0.0039,...,34995000000.0000,7694000000.0000,7650000000.0000,901000000.0000,0.1705,0.6206,4213000000.0000,2350303674000.0000,1.1120,28.1000,0.0000,258.6000,142.2700,280.4700,4.9000,0.1080,0.0693,0.2230,0.0500,0.1570,14.4000,7.4600,22.0000,0.9953,-0.0047,-0.4697,3.9000,0.3500,0.0189,0.0419
50%,2022-03-04 00:00:00,340.0000,14.5800,36.9200,35.8500,10.0000,38.0000,11.0000,0.0000,49.0000,2021-10-01 00:00:00,0.3661,0.6940,0.0012,-0.0533,0.1872,0.2658,393.9400,400.5000,390.3100,394.9400,109.5600,13125059.0000,0.0000,1.0000,0.0034,0.0034,0.0347,0.0727,0.1636,...,100131000000.0000,18383000000.0000,10320000000.0000,7520000000.0000,0.3516,7.8944,13210000000.0000,3052328976000.0000,1.2790,30.2000,0.0002,288.3500,161.3800,292.1500,7.2500,0.3010,0.1540,0.3440,0.2870,0.1800,19.7700,8.7700,77.0000,3.6119,1.2842,261.1927,36.9250,1.7000,0.0424,0.3371
75%,2024-06-21 00:00:00,1080.0000,85.8000,144.9700,142.7000,35.0000,147.2000,40.0000,10.0000,444.0000,2024-02-01 00:00:00,0.5027,0.9677,0.0044,-0.0170,0.8442,1.1379,1124.9000,1142.9700,1120.0300,1140.9900,172.1860,36914579.0000,0.0000,1.0000,0.0164,0.0163,0.0897,0.1955,0.3588,...,164529000000.0000,45374000000.0000,15955718000.0000,15320000000.0000,0.5952,66.2504,22677000000.0000,3825723245000.0000,1.5240,33.24

In [24]:
# Save full dataset (before bucket filter) to data/final/
out_full_parquet = FINAL / 'full_dataset.parquet'
out_full_csv     = FINAL / 'full_dataset.csv'

dataset.to_parquet(out_full_parquet, index=False)
dataset.to_csv(out_full_csv, index=False)

print(f'Full dataset saved.')
print(f'  Shape:   {dataset.shape}')
print(f'  Parquet: {out_full_parquet}')
print(f'  CSV:     {out_full_csv}')

Full dataset saved.
  Shape:   (927008, 101)
  Parquet: ../data/final/full_dataset.parquet
  CSV:     ../data/final/full_dataset.csv
